In [1]:
import numpy as np

## Desafio 1: Classificação Binária com Perceptron Treinável

In [2]:
# --- DADOS DO DESAFIO 1 ---
X_treino_fraude = np.array([
    [1.5, 1.0], [2.0, 2.0], [3.5, 1.5], [3.0, 3.0], 
    [6.5, 5.0], [7.0, 7.0], [8.5, 6.0], [9.0, 8.0]
])
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Novas transações para teste
transacao_A = np.array([2.5, 2.0])
transacao_B = np.array([8.0, 6.5])

# --- FUNÇÃO DE TREINAMENTO (Regra de Rosenblatt) ---
def treinar_perceptron(X, y, taxa_aprendizado=0.1, epocas=100):
    num_amostras, num_caracteristicas = X.shape
    pesos = np.zeros(num_caracteristicas) # Inicializa pesos com zero
    vies = 0.0                            # Inicializa viés com zero
    
    for _ in range(epocas):
        for i in range(num_amostras):
            # Calcula a soma ponderada: dot(x, w) + b
            soma = np.dot(X[i], pesos) + vies
            # Função degrau: 1 se soma >= 0, senão 0
            predicao = 1 if soma >= 0 else 0
            
            # Atualiza pesos apenas se errar (erro = y_real - predicao)
            erro = y[i] - predicao
            if erro != 0:
                pesos += taxa_aprendizado * erro * X[i]
                vies += taxa_aprendizado * erro
                
    return pesos, vies

# --- FUNÇÃO DE INFERÊNCIA ---
def prever_transacao(amostra, w, b):
    soma = np.dot(amostra, w) + b
    return 1 if soma >= 0 else 0

# --- EXECUÇÃO E ANÁLISE ---
pesos_finais, vies_final = treinar_perceptron(X_treino_fraude, y_treino_fraude)

print("--- RESULTADOS PERCEPTRON ---")
print(f"Pesos Finais: {pesos_finais}")
print(f"Viés Final: {vies_final}")
print(f"Classificação Transação A: {prever_transacao(transacao_A, pesos_finais, vies_final)} (0 = Legítima, 1 = Fraude)")
print(f"Classificação Transação B: {prever_transacao(transacao_B, pesos_finais, vies_final)} (0 = Legítima, 1 = Fraude)")

--- RESULTADOS PERCEPTRON ---
Pesos Finais: [0.05 0.1 ]
Viés Final: -0.7999999999999999
Classificação Transação A: 0 (0 = Legítima, 1 = Fraude)
Classificação Transação B: 1 (0 = Legítima, 1 = Fraude)


## Desafio 2: Predição de Risco de Churn com Classificador KNN

In [3]:
# --- DADOS DO DESAFIO 2 ---
X_treino_churn = np.array([
    [2.0, 1.0], [3.0, 0.0], [5.0, 1.0], [6.0, 2.0], 
    [15.0, 4.0], [18.0, 5.0], [20.0, 4.0], [22.0, 6.0]
])
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

cliente_1 = np.array([4.0, 1.0])
cliente_2 = np.array([17.0, 5.0])

# --- FUNÇÃO KNN ---
def classificar_knn(X_treino, y_treino, amostra, k=3, metrica="euclidiana"):
    # 1. Processamento vetorizado da distância
    if metrica == "euclidiana":
        distancias = np.sqrt(np.sum((X_treino - amostra)**2, axis=1))
    elif metrica == "manhattan":
        distancias = np.sum(np.abs(X_treino - amostra), axis=1)
        
    # 2. Ordena e pega os índices dos K mais próximos
    indices_vizinhos = np.argsort(distancias)[:k]
    
    # 3. Pega as classes dos vizinhos e as distâncias
    classes_vizinhos = y_treino[indices_vizinhos]
    dist_vizinhos = distancias[indices_vizinhos]
    
    # 4. Determinação da classe por maioria de votos
    # np.bincount conta as ocorrências; argmax pega a mais frequente
    classe_predita = np.bincount(classes_vizinhos).argmax()
    
    return classe_predita, indices_vizinhos, dist_vizinhos

# --- EXECUÇÃO E DIAGNÓSTICO ---
print("\n--- RESULTADOS KNN CHURN ---")
clientes_teste = [cliente_1, cliente_2]

for i, cliente in enumerate(clientes_teste):
    classe, idx_v, dist_v = classificar_knn(X_treino_churn, y_treino_churn, cliente, k=3, metrica="euclidiana")
    texto_risco = "Alto Risco" if classe == 1 else "Baixo Risco"
    
    print(f"\nCliente {i+1}:")
    print(f"- Classe Predita: {texto_risco}")
    print(f"- Índices dos vizinhos: {idx_v}")
    print(f"- Distâncias dos vizinhos: {dist_v}")


--- RESULTADOS KNN CHURN ---

Cliente 1:
- Classe Predita: Baixo Risco
- Índices dos vizinhos: [2 1 0]
- Distâncias dos vizinhos: [1.         1.41421356 2.        ]

Cliente 2:
- Classe Predita: Alto Risco
- Índices dos vizinhos: [5 4 6]
- Distâncias dos vizinhos: [1.         2.23606798 3.16227766]


## Desafio 3: Recomendação de Servidores Cloud

In [4]:
# --- DADOS DO DESAFIO 3 ---
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web", "Standard App Server", "Medium Backend & Cache",
    "Database Enterprise", "High Performance Computing", "Big Data & AI Training"
]

perfil_demandado = np.array([12.0, 28.0, 450.0])

# --- FUNÇÃO DE RECOMENDAÇÃO ---
def recomendar_servidor(catalogo, perfil, k=2):
    # Cálculo de distância euclidiana vetorizada
    distancias = np.sqrt(np.sum((catalogo - perfil)**2, axis=1))
    
    # Ordenação dos índices
    indices_proximos = np.argsort(distancias)[:k]
    
    return indices_proximos, distancias[indices_proximos]

# --- EXECUÇÃO E RELATÓRIO ---
idx_rec, dist_rec = recomendar_servidor(catalogo_servidores, perfil_demandado, k=2)

print("\n--- RESULTADOS RECOMENDAÇÃO CLOUD ---")
print("Requisitos exigidos: 12 vCPUs, 28GB RAM, 450GB SSD")
for pos, idx in enumerate(idx_rec):
    print(f"\n{pos+1}º Lugar:")
    print(f"- Máquina: {nomes_servidores[idx]}")
    specs = catalogo_servidores[idx]
    print(f"- Especificações: {specs[0]} vCPUs, {specs[1]}GB RAM, {specs[2]}GB SSD")
    print(f"- Distância Euclidiana: {dist_rec[pos]:.2f}")


--- RESULTADOS RECOMENDAÇÃO CLOUD ---
Requisitos exigidos: 12 vCPUs, 28GB RAM, 450GB SSD

1º Lugar:
- Máquina: Database Enterprise
- Especificações: 16.0 vCPUs, 32.0GB RAM, 500.0GB SSD
- Distância Euclidiana: 50.32

2º Lugar:
- Máquina: Medium Backend & Cache
- Especificações: 8.0 vCPUs, 16.0GB RAM, 250.0GB SSD
- Distância Euclidiana: 200.40
